# Módulo 03 · Aula 2 — Seaborn

**Capacitação Introdutória de Ciência de Dados · FEA.dev**

---

O Seaborn é uma camada construída **em cima** do Matplotlib. Ele não substitui o que
você aprendeu: ele encurta.

Três vantagens concretas:

1. **Fala DataFrame.** Você passa a tabela e os nomes das colunas, em vez de arrays.
2. **Agrupa sozinho.** O parâmetro `hue` separa por categoria e cuida da legenda.
3. **Faz estatística.** Alguns gráficos (boxplot, histograma com densidade, regressão)
   calculam o que precisam automaticamente.

Ao final desta aula você vai saber escolher e produzir os gráficos estatísticos que
aparecem em qualquer análise exploratória.

**Tempo estimado:** 60 minutos.

### Antes de começar — se você está no Google Colab

Este notebook lê arquivos da pasta `data/` do repositório, e no Colab a máquina começa vazia. **Execute a célula abaixo antes de qualquer outra**: ela traz o repositório e entra na pasta deste módulo, de modo que os caminhos `../data/...` usados no material funcionem sem alteração.

No VS Code ou no Jupyter local a célula não faz nada — os arquivos já estão no seu disco.

In [ ]:
# Setup do Google Colab.
# Traz o repositório da capacitação e entra na pasta deste módulo, para que os
# caminhos "../data/..." usados no material funcionem sem nenhuma alteração.
# Fora do Colab (VS Code, Jupyter local) esta célula não faz nada.
# Pode ser executada mais de uma vez sem problema.
import os
import subprocess
import sys

PASTA_DESTE_MODULO = "03_Visualizacao_EDA"
REPOSITORIO = "https://github.com/gustavokatsuo/Introducao-a-Ciencia-de-Dados.git"

if "google.colab" in sys.modules and not os.path.isdir("../data"):
    destino = "/content/Introducao-a-Ciencia-de-Dados"
    if not os.path.isdir(destino):
        print("Baixando o material da capacitação...")
        subprocess.run(["git", "clone", "--depth", "1", REPOSITORIO, destino], check=True)
    os.chdir(os.path.join(destino, PASTA_DESTE_MODULO))
    print("Pronto. Pasta de trabalho:", os.getcwd())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

%matplotlib inline

# set_theme define a aparência de TODOS os gráficos daqui em diante,
# inclusive os feitos com matplotlib puro.
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (9, 5)

print("seaborn:", sns.__version__)

In [ ]:
from IPython.display import Image
Image("../assets/escolha_do_grafico.png", width=760)

## 1. Preparando os dados

Vamos usar duas bases: os preços das ações (cruzados com o cadastro das empresas) e o
cadastro de clientes — este último **limpo** com o procedimento da aula 02.4, condensado
em uma função.

In [ ]:
acoes = pd.read_csv("../data/acoes_b3.csv", parse_dates=["data"])
empresas = pd.read_csv("../data/empresas_b3.csv")

acoes = acoes.merge(empresas, on="ticker", how="left", validate="m:1")
acoes["ano"] = acoes["data"].dt.year

# Retorno diário por ativo. groupby + pct_change garante que o cálculo não
# "atravesse" de um ativo para o outro.
acoes = acoes.sort_values(["ticker", "data"])
acoes["retorno_diario"] = acoes.groupby("ticker")["fechamento_ajustado"].pct_change() * 100

acoes[["data", "ticker", "setor", "fechamento_ajustado", "retorno_diario"]].head()

In [ ]:
def carregar_clientes_limpos(caminho="../data/clientes_corretora.csv"):
    """Carrega a base de clientes aplicando a limpeza da aula 02.4."""
    df = pd.read_csv(caminho).drop_duplicates()

    # dinheiro em formato brasileiro -> float
    patrimonio = (
        df["patrimonio_investido"].astype(str)
        .str.replace("R$", "", regex=False)
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
        .str.strip()
    )
    df["patrimonio_investido"] = pd.to_numeric(patrimonio, errors="coerce")

    # categorias padronizadas
    df["perfil_investidor"] = df["perfil_investidor"].str.strip().str.title()
    df["cidade"] = (
        df["cidade"].astype(str).str.strip()
        .str.normalize("NFKD").str.encode("ascii", errors="ignore").str.decode("utf-8")
        .str.upper()
    )
    df["ativo"] = df["ativo"].astype(str).str.lower().map(
        {"sim": True, "nao": False, "1": True, "0": False}
    )

    # idades impossíveis viram faltantes e são preenchidas pela mediana
    df.loc[~df["idade"].between(18, 110), "idade"] = np.nan
    df["idade"] = df["idade"].fillna(df["idade"].median()).astype(int)

    return df


clientes = carregar_clientes_limpos()
print(clientes.shape)
clientes.head(3)

## 2. Distribuição de uma variável

### `histplot`

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))

sns.histplot(data=clientes, x="patrimonio_investido", bins=40, ax=ax)

ax.set_title("Distribuição do patrimônio investido")
ax.set_xlabel("Patrimônio investido (R$)")
ax.set_ylabel("Número de clientes")

plt.show()

A distribuição é **assimétrica à direita**: a maioria dos clientes tem patrimônio baixo
e uma minoria tem valores muito altos, esticando a cauda. Esse formato é a regra em
variáveis de dinheiro — renda, patrimônio, faturamento, capitalização de mercado.

Consequência prática imediata: **a média não representa o cliente típico.** Ela é puxada
pelos poucos muito ricos. A mediana descreve melhor. Voltaremos a isso na próxima aula.

In [ ]:
# hue separa por categoria, e o Seaborn cuida de cores e legenda
fig, ax = plt.subplots(figsize=(9.5, 4.5))

sns.histplot(data=clientes, x="patrimonio_investido", hue="perfil_investidor",
             bins=35, element="step", ax=ax)

ax.set_title("Patrimônio investido por perfil de investidor")
ax.set_xlabel("Patrimônio investido (R$)")
ax.set_ylabel("Número de clientes")

plt.show()

In [ ]:
# kde=True acrescenta uma curva de densidade suavizada
fig, ax = plt.subplots(figsize=(9, 4.5))

sns.histplot(data=acoes, x="retorno_diario", bins=80, kde=True, ax=ax)

ax.set_title("Distribuição dos retornos diários (todos os ativos, 2021–2025)")
ax.set_xlabel("Retorno diário (%)")
ax.set_ylabel("Número de observações")
ax.axvline(0, color="black", linewidth=1)

plt.show()

Retornos diários têm um formato característico: concentrados perto de zero, quase
simétricos, e com **caudas mais grossas** que uma distribuição normal — dias de variação
extrema são raros, mas bem mais frequentes do que a curva do sino previria. Essa
observação sustenta boa parte da literatura de risco em finanças.

### `boxplot`

O boxplot resume uma distribuição em cinco números e mostra os extremos. Ele é o gráfico
ideal para **comparar distribuições entre grupos**, porque ocupa pouco espaço.

Como ler:

```
          ┌───┬───────┐
   ├──────┤   │       ├──────┤        ● ●
          └───┴───────┘
   │      │   │       │      │        │
  mín.   Q1  mediana Q3    máx.    pontos fora
(dentro do limite)                (possíveis outliers)
```

A **caixa** contém os 50% centrais dos dados (do 1º ao 3º quartil). A linha dentro dela
é a **mediana**. As hastes vão até 1,5× a altura da caixa; o que passa disso vira ponto.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ordem = acoes.groupby("ticker")["retorno_diario"].std().sort_values().index

sns.boxplot(data=acoes, x="ticker", y="retorno_diario", order=ordem, ax=ax)

ax.set_title("Dispersão dos retornos diários por ativo (ordenado por volatilidade)")
ax.set_xlabel("")
ax.set_ylabel("Retorno diário (%)")
ax.axhline(0, color="black", linewidth=0.8)

plt.show()

Este gráfico responde a uma pergunta de risco em um segundo: **quais ativos oscilam
mais**. Caixas mais altas significam maior volatilidade. Repare que MGLU3 tem a caixa
mais alta e os pontos extremos mais distantes — coerente com a trajetória que vimos na
aula anterior.

> A ordenação (`order=`) não é enfeite: ela transforma um amontoado de caixas em um
> ranking legível.

## 3. Comparando categorias

### `barplot` e `countplot`

- `countplot` conta **linhas** por categoria;
- `barplot` mostra uma **estatística** (por padrão, a média) de uma variável numérica
  por categoria.

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(13, 4.5))

sns.countplot(data=clientes, x="perfil_investidor", ax=eixos[0])
eixos[0].set_title("Quantos clientes em cada perfil")
eixos[0].set_xlabel("")
eixos[0].set_ylabel("Número de clientes")

sns.barplot(data=clientes, x="perfil_investidor", y="patrimonio_investido",
            estimator="median", errorbar=None, ax=eixos[1])
eixos[1].set_title("Patrimônio mediano por perfil")
eixos[1].set_xlabel("")
eixos[1].set_ylabel("Patrimônio mediano (R$)")

fig.tight_layout()
plt.show()

> **Atenção:** Note o `estimator="median"`. **Por padrão o `barplot` mostra a média** — e,
> com patrimônio, a média engana pelo que vimos no histograma. Sempre saiba qual
> estatística a sua barra está mostrando, e diga isso no rótulo do eixo.

In [ ]:
# Duas categorias ao mesmo tempo, com hue
fig, ax = plt.subplots(figsize=(11, 5))

sns.barplot(data=clientes, x="estado", y="patrimonio_investido",
            hue="perfil_investidor", estimator="median", errorbar=None, ax=ax)

ax.set_title("Patrimônio mediano por estado e perfil")
ax.set_xlabel("Estado")
ax.set_ylabel("Patrimônio mediano (R$)")
ax.legend(title="Perfil")

plt.show()

> Quando um grupo tem poucos casos, a barra correspondente vira ruído — a "mediana" de
> três clientes não descreve nada. **Sempre confira o tamanho dos grupos** antes de
> interpretar um gráfico como este:

In [ ]:
clientes.groupby(["estado", "perfil_investidor"]).size().unstack(fill_value=0)

## 4. Relação entre variáveis

### `scatterplot`

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))

sns.scatterplot(data=clientes, x="patrimonio_investido", y="aporte_mensal",
                hue="perfil_investidor", alpha=0.7, ax=ax)

ax.set_title("Aporte mensal × patrimônio investido")
ax.set_xlabel("Patrimônio investido (R$)")
ax.set_ylabel("Aporte mensal (R$)")
ax.legend(title="Perfil")

plt.show()

In [ ]:
# regplot acrescenta a reta de regressão — útil para enxergar a tendência
fig, ax = plt.subplots(figsize=(8, 5))

sns.regplot(data=clientes, x="idade", y="patrimonio_investido",
            scatter_kws={"alpha": 0.4, "s": 22}, line_kws={"color": "#c0392b"}, ax=ax)

ax.set_title("Patrimônio investido × idade")
ax.set_xlabel("Idade (anos)")
ax.set_ylabel("Patrimônio investido (R$)")

plt.show()

> **Atenção:** A reta sempre aparece, mesmo quando não há relação nenhuma. Ela é o
> resultado de um cálculo, não uma prova. Olhe a **nuvem de pontos**, não a reta: se os
> pontos estão espalhados sem padrão, a inclinação é irrelevante por mais convincente que
> pareça.

### `lineplot`

In [ ]:
media_mensal = (
    acoes.assign(ano_mes=acoes["data"].dt.to_period("M").dt.to_timestamp())
    .groupby(["ano_mes", "setor"], as_index=False)["retorno_diario"].mean()
)

fig, ax = plt.subplots(figsize=(12, 5))

sns.lineplot(data=media_mensal, x="ano_mes", y="retorno_diario",
             hue="setor", linewidth=1.4, ax=ax)

ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Retorno diário médio por mês e setor")
ax.set_xlabel("Mês")
ax.set_ylabel("Retorno diário médio (%)")
ax.legend(title="Setor", bbox_to_anchor=(1.02, 1), loc="upper left")

fig.tight_layout()
plt.show()

### `heatmap`: matriz de correlação

Para ver **todas as relações de uma vez**, calcule a matriz de correlação e pinte-a. É
o gráfico que abre praticamente toda análise exploratória com muitas variáveis
numéricas.

In [ ]:
# Retornos diários lado a lado, um ativo por coluna
retornos = acoes.pivot_table(index="data", columns="ticker", values="retorno_diario")
correlacao = retornos.corr()

fig, ax = plt.subplots(figsize=(8, 6.5))

sns.heatmap(correlacao, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.5,
            cbar_kws={"label": "correlação"}, ax=ax)

ax.set_title("Correlação entre os retornos diários (2021–2025)", pad=14)
ax.set_xlabel("")
ax.set_ylabel("")

plt.show()

Três decisões importantes nesse `heatmap`:

- **`cmap="RdBu_r"` com `center=0`** — uma escala divergente, porque a correlação tem
  um ponto neutro natural no zero. Usar uma escala sequencial aqui esconderia o sinal;
- **`vmin=-1, vmax=1`** — fixa a escala nos limites teóricos, para que a cor signifique
  a mesma coisa em qualquer versão do gráfico;
- **`annot=True`** — mostra os números, porque cor sozinha não permite ler valores
  exatos.

E a leitura: o par mais correlacionado é **ITUB4–BBDC4** (0,71) — dois bancos de varejo,
expostos aos mesmos fatores. B3SA3 acompanha os dois com intensidade intermediária
(cerca de 0,5), e **VALE3 é o ativo menos correlacionado com todos os outros** (0,09 a
0,24), o que faz sentido: seu resultado depende muito mais do preço do minério e da
demanda chinesa do que do ciclo doméstico.

Esse é o argumento quantitativo da diversificação: juntar ativos pouco correlacionados
reduz a oscilação da carteira mais do que juntar ativos parecidos. Repare também que
**nenhuma correlação é negativa** — em um mercado em queda geral, tudo tende a cair
junto, e a diversificação dentro de uma única bolsa tem limite.

### `pairplot`: tudo contra tudo

Combina histogramas na diagonal com dispersões fora dela. Serve para uma primeira varredura
— desde que sejam poucas variáveis, senão vira ilegível.

In [ ]:
grade = sns.pairplot(
    clientes[["idade", "patrimonio_investido", "aporte_mensal", "perfil_investidor"]],
    hue="perfil_investidor",
    height=2.2,
    plot_kws={"alpha": 0.55, "s": 18},
)
grade.figure.suptitle("Relações entre as variáveis do cadastro de clientes", y=1.02)

plt.show()

## 5. Aparência

`sns.set_theme()` muda o estilo de todos os gráficos. Os estilos disponíveis são
`whitegrid`, `darkgrid`, `white`, `dark` e `ticks`.

In [ ]:
for estilo in ["whitegrid", "darkgrid", "ticks"]:
    with sns.axes_style(estilo):          # aplica o estilo só dentro deste bloco
        fig, ax = plt.subplots(figsize=(5, 2.6))
        sns.histplot(data=clientes, x="idade", bins=20, ax=ax)
        ax.set_title(f'style="{estilo}"')
        ax.set_xlabel("Idade")
        ax.set_ylabel("")
        plt.show()

In [ ]:
# Paletas: qualitativa para categorias, sequencial para ordem, divergente para
# valores com um centro natural (como a correlação, centrada em zero).
for nome in ["deep", "viridis", "RdBu_r"]:
    print(f"{nome:>8}:", sns.color_palette(nome, 6).as_hex())

sns.set_theme(style="whitegrid", palette="deep")     # volta ao padrão da aula

## 6. Matplotlib ou Seaborn?

Não é escolha: use os dois. O padrão de trabalho é

1. **Seaborn desenha** — `sns.algumacoisa(data=..., x=..., y=..., hue=..., ax=ax)`;
2. **Matplotlib ajusta** — `ax.set_title(...)`, `ax.set_ylabel(...)`, `ax.axhline(...)`.

Todo gráfico do Seaborn devolve (ou aceita) um `ax` do Matplotlib. Por isso tudo que
você aprendeu na aula anterior continua valendo.

| Situação | Ferramenta |
|---|---|
| Gráfico estatístico rápido, com agrupamento por categoria | Seaborn |
| Boxplot, heatmap, pairplot, regressão | Seaborn |
| Controle fino de anotações, eixos e layout | Matplotlib |
| Combinação incomum de elementos | Matplotlib |

## 7. Recapitulando

- `sns.set_theme(style=..., palette=...)` define a aparência de tudo.
- Distribuição: `histplot` (com `kde=True`, `hue`, `element="step"`), `boxplot` para
  comparar grupos.
- Categorias: `countplot` (conta linhas), `barplot` (estatística — **confira se é média
  ou mediana**).
- Relações: `scatterplot`, `regplot` (com a reta), `lineplot` (evolução), `heatmap`
  (matriz de correlação), `pairplot` (varredura inicial).
- Todo gráfico aceita `ax=`: desenhe com Seaborn, ajuste com Matplotlib.
- Antes de interpretar um agrupamento, **confira o tamanho dos grupos**.

**Próxima aula:** estatística descritiva — os números por trás desses gráficos.